In [ ]:
import pandas as pd
from pathlib import Path

RAW_PATH = Path("data/raw/08_investor_transactions.csv")
OUT_PATH = Path("data/processed/08_investor_transactions_clean.csv")

df = pd.read_csv(RAW_PATH)
print("Before cleaning:", df.shape)

# 1. Parse transaction_date
df["transaction_date"] = pd.to_datetime(df["transaction_date"], errors="coerce")
bad_dates = df["transaction_date"].isnull().sum()
print(f"Rows with unparseable dates: {bad_dates}")

# 2. transaction_type and kyc_status already clean (verified above) — no remapping needed

# 3. Validate amount_inr > 0
invalid_amount = df[df["amount_inr"] <= 0]
print(f"Rows with amount_inr <= 0: {len(invalid_amount)}")
if len(invalid_amount) > 0:
    print(invalid_amount[["investor_id", "transaction_type", "amount_inr"]])

# 4. Check for duplicate transactions (same investor, date, amount, type)
dupes = df.duplicated(subset=["investor_id", "transaction_date", "amfi_code", "amount_inr", "transaction_type"])
print(f"Duplicate transactions found: {dupes.sum()}")
df = df[~dupes]

print("After cleaning:", df.shape)

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_PATH, index=False)
print(f"Saved -> {OUT_PATH}")

In [ ]:
import pandas as pd
from pathlib import Path

RAW_PATH = Path("data/raw/02_nav_history.csv")
OUT_PATH = Path("data/processed/02_nav_history_clean.csv")

df = pd.read_csv(RAW_PATH)
print("Before cleaning:", df.shape)

# 1. Parse dates
df["date"] = pd.to_datetime(df["date"])

# 2. Sort by amfi_code, then date
df = df.sort_values(["amfi_code", "date"]).reset_index(drop=True)

# 3. Forward-fill missing NAV per fund (holidays/weekends)
df["nav"] = df.groupby("amfi_code")["nav"].ffill()

# 4. Remove duplicate (amfi_code, date) rows
before_dupes = len(df)
df = df.drop_duplicates(subset=["amfi_code", "date"])
print(f"Removed {before_dupes - len(df)} duplicate rows")

# 5. Validate NAV > 0
invalid = df[df["nav"] <= 0]
print(f"Rows with NAV <= 0: {len(invalid)}")
if len(invalid) > 0:
    print(invalid)

print("After cleaning:", df.shape)

# Save
OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_PATH, index=False)
print(f"Saved -> {OUT_PATH}")

In [ ]:
import pandas as pd
from pathlib import Path

RAW_PATH = Path("data/raw/07_scheme_performance.csv")
OUT_PATH = Path("data/processed/07_scheme_performance_clean.csv")

df = pd.read_csv(RAW_PATH)
print("Before cleaning:", df.shape)

# 1. Confirm all return/ratio columns are genuinely numeric (already float64, but double-check for NaNs introduced by bad values)
numeric_cols = [
    "return_1yr_pct", "return_3yr_pct", "return_5yr_pct", "benchmark_3yr_pct",
    "alpha", "beta", "sharpe_ratio", "sortino_ratio", "std_dev_ann_pct",
    "max_drawdown_pct", "aum_crore", "expense_ratio_pct"
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

nulls_after = df[numeric_cols].isnull().sum()
print("\nNulls introduced by numeric coercion (should be 0 if already clean):")
print(nulls_after[nulls_after > 0])

# 2. Flag anomalies — negative AUM, absurd returns, negative expense ratio
anomalies = df[(df["aum_crore"] < 0) | (df["expense_ratio_pct"] < 0) | (df["return_1yr_pct"].abs() > 200)]
print(f"\nFlagged anomaly rows: {len(anomalies)}")
if len(anomalies) > 0:
    print(anomalies[["amfi_code", "scheme_name", "aum_crore", "expense_ratio_pct", "return_1yr_pct"]])

# 3. Validate expense_ratio_pct is within 0.1% - 2.5%
out_of_range = df[(df["expense_ratio_pct"] < 0.1) | (df["expense_ratio_pct"] > 2.5)]
print(f"\nRows with expense_ratio_pct outside 0.1%-2.5%: {len(out_of_range)}")
if len(out_of_range) > 0:
    print(out_of_range[["amfi_code", "scheme_name", "expense_ratio_pct"]])

print("\nAfter cleaning:", df.shape)

OUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(OUT_PATH, index=False)
print(f"Saved -> {OUT_PATH}")

In [ ]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path("data/raw")
OUT_DIR = Path("data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Files not covered by the detailed Day 2 cleaning tasks — generic pass only
FILES = [
    "01_fund_master.csv",
    "03_aum_by_fund_house.csv",
    "04_monthly_sip_inflows.csv",
    "05_category_inflows.csv",
    "06_industry_folio_count.csv",
    "09_portfolio_holdings.csv",
    "10_benchmark_indices.csv",
]

def clean_file(filename: str):
    path = RAW_DIR / filename
    df = pd.read_csv(path)
    print(f"\n{'='*60}\n{filename}\n{'='*60}")
    print(f"Before: {df.shape}")

    # Parse any column that looks like a date
    date_cols = [c for c in df.columns if "date" in c.lower() or c.lower() == "month"]
    for col in date_cols:
        try:
            df[col] = pd.to_datetime(df[col], errors="coerce")
            bad = df[col].isnull().sum()
            if bad > 0:
                print(f"  '{col}': {bad} unparseable date values")
        except Exception as e:
            print(f"  Could not parse '{col}' as date: {e}")

    # Remove exact duplicate rows
    before = len(df)
    df = df.drop_duplicates()
    removed = before - len(df)
    if removed > 0:
        print(f"  Removed {removed} exact duplicate rows")

    # Flag negative values in numeric columns (informational only — not auto-removed)
    numeric_cols = df.select_dtypes(include="number").columns
    for col in numeric_cols:
        neg_count = (df[col] < 0).sum()
        if neg_count > 0:
            print(f"  '{col}': {neg_count} negative values — review manually")

    # Check for nulls
    null_counts = df.isnull().sum()
    nulls_found = null_counts[null_counts > 0]
    if len(nulls_found) > 0:
        print(f"  Null values found:\n{nulls_found}")

    print(f"After: {df.shape}")

    out_name = filename.replace(".csv", "_clean.csv")
    out_path = OUT_DIR / out_name
    df.to_csv(out_path, index=False)
    print(f"Saved -> {out_path}")

if __name__ == "__main__":
    for f in FILES:
        clean_file(f)
    print("\nAll 7 files processed.")
